In [22]:
pip install selenium

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [2]:
# Set your ChromeDriver path (use raw string to avoid escape issues)
chrome_path = r"C:\Users\haika\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"

# Setup Selenium with Chrome
service = Service(chrome_path)
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in background
driver = webdriver.Chrome(service=service, options=options)

In [3]:
# Target page
url = "https://www.worldaquatics.com/competitions/4864/67th-international-divers-day/results?event=ac05a075-5a4e-420a-b84d-91c95cf647d9"
driver.get(url)
time.sleep(10)  # Give the page time to fully load

In [4]:
tables = driver.find_elements(By.TAG_NAME, "table")
expand_buttons = driver.find_elements(By.CLASS_NAME, "js-results-row-expand")

In [5]:
for btn in expand_buttons:
    try:
        driver.execute_script("arguments[0].click();", btn)  # Use JS to avoid visibility issues
        time.sleep(1)  # Small wait after each click to allow sub-table to render
    except Exception as e:
        print("⚠️ Couldn't click expand button:", e)

In [6]:
# 2. Now find the sub-tables
sub_tables = driver.find_elements(By.CLASS_NAME, "results-table__sub-table")

# Initialize
data = []
target_names = ["Michelle HEIMBERG", "Grace REID"]
csv_headers = [
    "Athlete", "Dive Number", "Dive Description", "DD (Degree of Difficulty)",
    "Judge 1", "Judge 2", "Judge 3", "Judge 4", "Judge 5","Judge 6", "Judge 7",
    "Dive Points", "Total Points"
]

def decode_dive_code(code):
    dive_types = {
        '1': "Forward",
        '2': "Back",
        '3': "Reverse",
        '4': "Inward",
        '5': "Twisting"
    }

    positions = {
        'A': "Straight",
        'B': "Pike",
        'C': "Tuck",
        'D': "Free"
    }

    try:
        if code.startswith("5"):  # Twisting dives: 4-digit + position
            takeoff = code[1]
            somersaults = int(code[2]) + 0.5  # 5 = 2.5
            twists = int(code[3]) * 0.5       # 2 = 1 twist
            position = positions.get(code[4], "Unknown")
            dive_type = dive_types.get(takeoff, "Unknown")
            return f"{dive_type} {somersaults} Somersaults with {twists} Twist(s), {position}"
        else:
            type_digit = code[0]
            somersaults = int(code[1:3]) * 0.5
            position = positions.get(code[-1], "Unknown")
            dive_type = dive_types.get(type_digit, "Unknown")
            return f"{dive_type} {somersaults} Somersaults, {position}"
    except:
        return "Unknown Dive"

In [7]:
# Search and parse table that contains our target athletes
for i, table in enumerate(tables):
    table_text = table.text

    if "Michelle" in table_text or "Grace" in table_text:
        lines = table_text.splitlines()
        i = 0
        while i < len(lines) - 10:
            try:
                full_name = f"{lines[i+2]} {lines[i+3]}"
                if full_name in target_names:
                    print(f"✅ Match: {full_name}")
                    # Sub-table headers start after the age/points row
                    dive_start_idx = i + 6

                    # Read until next athlete or end of data
                    while dive_start_idx < len(lines):
                        dive_line = lines[dive_start_idx]
                        tokens = dive_line.split()

                        # Dive rows typically start with a number and a dive code
                        if len(tokens) >= 13 and tokens[0].isdigit():
                            print(tokens)
                            dive_number = tokens[0]
                            dive_code = tokens[1]
                            dive_desc = f"{dive_code} - {decode_dive_code(dive_code)}"
                            dd = tokens[2]
                            judges = tokens[3:10]  # J1 to J7
                            dive_points = tokens[10]
                            total_points = tokens[11]

                            data.append({
                                "Athlete": full_name,
                                "Dive Number": dive_number,
                                "Dive Description": dive_desc,
                                "DD (Degree of Difficulty)": dd,
                                "Judge 1": judges[0],
                                "Judge 2": judges[1],
                                "Judge 3": judges[2],
                                "Judge 4": judges[3],
                                "Judge 5": judges[4],
                                "Judge 6": judges[5],
                                "Judge 7": judges[6],
                                "Dive Points": dive_points,
                                "Total Points": total_points
                            })
                            dive_start_idx += 1
                        else:
                            break
            except Exception as e:
                print(f"❌ Error parsing athlete section: {e}")
            i += 1
        break  # Stop after first matching table

✅ Match: Michelle HEIMBERG
['1', '405B', '3.0', '4.5', '4.0', '4.5', '4.5', '4.5', '4.5', '4.5', '40.50', '40.50', '6', '6', '22.50']
['2', '107B', '3.1', '7.0', '7.0', '6.5', '6.0', '6.5', '6.5', '6.5', '60.45', '100.95', '3', '5', '23.55']
['3', '205B', '3.0', '7.0', '7.5', '7.0', '7.5', '8.0', '7.5', '8.0', '67.50', '168.45', '1', '3', '13.05']
['4', '305B', '3.0', '8.0', '7.5', '7.0', '7.0', '7.5', '7.0', '8.0', '66.00', '234.45', '1', '1', '-']
['5', '5152B', '3.0', '7.0', '6.5', '7.0', '6.5', '7.5', '6.0', '7.0', '61.50', '295.95', '3', '1', '-']
✅ Match: Grace REID
['1', '405B', '3.0', '6.0', '6.0', '6.0', '6.0', '6.5', '6.5', '6.5', '55.50', '55.50', '4', '4', '7.50']
['2', '107B', '3.1', '6.5', '6.5', '6.0', '5.5', '5.5', '6.5', '5.5', '55.80', '111.30', '4', '3', '13.20']
['3', '305B', '3.0', '6.0', '6.5', '5.5', '7.5', '7.0', '6.0', '6.5', '57.00', '168.30', '=2', '4', '13.20']
['4', '205B', '3.0', '6.0', '6.0', '6.0', '6.5', '6.5', '6.5', '6.5', '57.00', '225.30', '2', '3',

In [8]:
if data:
    df = pd.DataFrame(data, columns=csv_headers)
    df.to_csv(r"C:\Users\haika\Desktop\Document Kerja\ISN Application Document\diving_results.csv", index=False)
    print("\n📁 Dive data saved to diving_results.csv")
    print(df)
else:
    print("⚠️ No dive data extracted.")


📁 Dive data saved to diving_results.csv
             Athlete Dive Number  \
0  Michelle HEIMBERG           1   
1  Michelle HEIMBERG           2   
2  Michelle HEIMBERG           3   
3  Michelle HEIMBERG           4   
4  Michelle HEIMBERG           5   
5         Grace REID           1   
6         Grace REID           2   
7         Grace REID           3   
8         Grace REID           4   
9         Grace REID           5   

                                    Dive Description  \
0                405B - Inward 2.5 Somersaults, Pike   
1               107B - Forward 3.5 Somersaults, Pike   
2                  205B - Back 2.5 Somersaults, Pike   
3               305B - Reverse 2.5 Somersaults, Pike   
4  5152B - Forward 5.5 Somersaults with 1.0 Twist...   
5                405B - Inward 2.5 Somersaults, Pike   
6               107B - Forward 3.5 Somersaults, Pike   
7               305B - Reverse 2.5 Somersaults, Pike   
8                  205B - Back 2.5 Somersaults, Pike   
9 